In [16]:
import os
import pprint
from dotenv import load_dotenv,find_dotenv
from langchain_core.messages import HumanMessage
from langchain_openai import AzureChatOpenAI,AzureOpenAIEmbeddings
from langgraph.graph import StateGraph,MessagesState
from langgraph.prebuilt import  ToolNode,tools_condition
from langchain_core.tools import tool
from IPython.display import Image
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(find_dotenv(),override=True)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version="2025-01-01-preview"


In [17]:
@tool
def get_weather(location:str)->str:
    """Get the current weather for a given location."""
    return f"The current weather in {location} is sunny with a temperature of 25°C."

tools=[get_weather]

llm = AzureChatOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version=api_version
    ).bind_tools(tools)

In [18]:
def chatbot(state:MessagesState):
    """it will also call tools node"""
    res=llm.invoke(state["messages"])
    return {"messages":res}

builder=StateGraph(MessagesState)
builder.add_node("chatbot",chatbot) 
tool_node=ToolNode(tools)
builder.add_node("tools",tool_node)

builder.set_entry_point("chatbot")
builder.add_conditional_edges(
    "chatbot",
    tools_condition)
builder.add_edge("tools","chatbot")

checkpoint=MemorySaver()
graph=builder.compile( checkpointer=checkpoint, interrupt_before=["tools"])
# display(Image(graph.get_graph().draw_mermaid_png()))

config={"configurable":{"thread_id":"1"}}
intput_message=HumanMessage(content="What's the weather like in japan?")
res=graph.invoke({"messages":[intput_message]},config=config)
# res["messages"][-1].content

In [28]:
snapshot = graph.get_state(config)
existing=snapshot.values["messages"][-1]
existing


AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_fCwmSNZQuKZH4Tvcm7n5zhBr', 'function': {'arguments': '{"location":"Japan"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_e05894342c', 'id': 'chatcmpl-DDyAiWLEKh7ATzCMXTaeEIHs7mIt6', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'tool_c

In [ ]:
graph.invoke(None,config=config)
